# Citation-Grounded Legal Document Research Assistant
## Interactive Pipeline Demonstration & Technical Walkthrough

This notebook demonstrates the step-by-step execution of the **Retrieval-Augmented Generation (RAG)** pipeline designed for legal documents:
1. **Document Ingestion & PDF Extraction**
2. **Semantic Paragraph Chunking & Legal Metadata Extraction**
3. **Dense Vector Embeddings & Similarity Indexing**
4. **Top-$k$ Semantic Retrieval with Cosine Similarity**
5. **Context Assembly & Grounded Prompt Construction**
6. **Citation Mapping & Verifiable Passage Inspection**

In [ ]:
import os
import sys

# Set up system path to import backend modules
project_root = os.path.abspath('..')
backend_dir = os.path.join(project_root, 'backend')
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

from app.config.config import settings
from app.services.extraction_service import ExtractionService
from app.services.chunking_service import ChunkingService
from app.nlp.embeddings import EmbeddingService
from app.services.retrieval_service import RetrievalService
from app.services.rag_service import RAGService
from app.rag.context_builder import ContextBuilder

print("All backend modules imported successfully!")
print(f"Project root: {project_root}")

### Step 1: Document Extraction (PDF & Plain Text)
Extract layout-aware pages from authentic Supreme Court judgments.

In [ ]:
extractor = ExtractionService()
pdf_path = os.path.join(project_root, 'data', 'documents', 'raw', 'gurbaksh_singh_sibbia_1980.pdf')

pages = extractor.extract_from_file(pdf_path)
print(f"Extracted {len(pages)} page(s) from {os.path.basename(pdf_path)}")
print("\n--- First 300 characters of Page 1 ---")
print(pages[0]['text'][:300] + "...")

### Step 2: Legal Paragraph Chunking & Section Detection
Preserve parent document ID, section/paragraph reference (e.g., `Para 15`, `Section 438`), and page number.

In [ ]:
chunker = ChunkingService()
chunks = chunker.chunk_document(
    document_id="sibbia-1980",
    document_title="Gurbaksh Singh Sibbia v. State of Punjab",
    pages=pages
)

print(f"Document segmented into {len(chunks)} structured chunks.")
sample_chunk = chunks[3]
print(f"\nSample Chunk Index: {sample_chunk.chunk_index}")
print(f"Reference: {sample_chunk.section_ref}")
print(f"Text snippet: {sample_chunk.text[:250]}...")

### Step 3: Semantic Embeddings & Retrieval
Convert user questions into dense vector representations and retrieve the top-5 most relevant legal chunks.

In [ ]:
retrieval_service = RetrievalService()
user_query = "What factors are considered while granting anticipatory bail?"

retrieved_chunks = retrieval_service.retrieve_top_k(user_query, k=3)
print(f"Retrieved {len(retrieved_chunks)} relevant chunks for query: '{user_query}'\n")

for i, chk in enumerate(retrieved_chunks, start=1):
    print(f"[{i}] Document: {chk.document_title}")
    print(f"    Section: {chk.section_ref}")
    print(f"    Similarity Score: {chk.similarity_score}")
    print(f"    Snippet: {chk.text[:150]}...\n")

### Step 4: Grounded Answer Synthesis & Citation Verification
Execute the full RAG pipeline to generate a grounded answer with inline citations `[1]`, `[2]`.

In [ ]:
rag_service = RAGService()
result = rag_service.process_query(user_query, top_k=3)

print("=== GROUNDED LEGAL ANSWER ===\n")
print(result.answer)
print("\n=== INLINE VERIFIABLE CITATIONS ===")
for cit in result.citations:
    print(f"Citation [{cit.marker_index}]: {cit.document} ({cit.section})")
    print(f"Verbatim Snippet: \"{cit.snippet}\"")
    print(f"Match Confidence: {cit.confidence_score * 100:.1f}%\n")

print(f"Confidence Status: {result.confidence}")
print(f"Execution Latency: {result.latency_ms} ms")

### Step 5: Hallucination Prevention / Negative Test Case
Demonstrating that the system refuses to speculate on out-of-domain queries.

In [ ]:
out_of_domain_query = "How do you bake a chocolate brownie cake?"
neg_result = rag_service.process_query(out_of_domain_query)

print(f"Query: {out_of_domain_query}")
print(f"Answer: {neg_result.answer}")
print(f"Confidence: {neg_result.confidence}")
print(f"Citations count: {len(neg_result.citations)}")